# Парсинг Netflix: runtime, IMDb votes, revenue, budget

Финальный пайплайн: IMDb собирает `runtime`, `vote_average_imdb`, `vote_count_imdb`, `budget`; Box Office Mojo собирает `revenue` и fallback для `budget`; The Numbers собирает `Infl. Adj. Dom. BO` и fallback для `Production Budget`.

### Импорты

In [1]:
import pandas as pd
import numpy as np
import re
import time
import json

from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException, WebDriverException

### Загружаю датасет

In [123]:
df = pd.read_csv("netflix_enriched_2.csv")

df.shape

(498, 23)

### Проверяю пропуски

In [124]:
df[["runtime", "vote_count_imdb", "vote_average_imdb", "revenue", "budget"]].isna().sum()

runtime              184
vote_count_imdb      283
vote_average_imdb    283
revenue              395
budget               390
dtype: int64

### Привожу нужные колонки к числам

In [125]:
df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce")
df["vote_count_imdb"] = pd.to_numeric(df["vote_count_imdb"], errors="coerce")
df["vote_average_imdb"] = pd.to_numeric(df["vote_average_imdb"], errors="coerce")
df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce")
df["budget"] = pd.to_numeric(df["budget"], errors="coerce")

In [126]:
if "genres" not in df.columns:
    df["genres"] = np.nan

if "revenue_adj_domestic" not in df.columns:
    df["revenue_adj_domestic"] = np.nan

### Маски пропусков

In [127]:
imdb_parse = df.loc[
    df["imdb_id"].notna(),
    ["title", "release_year", "imdb_id", "runtime", "vote_average_imdb", "vote_count_imdb", "budget", "genres"]
].copy()

mojo_parse = df.loc[
    df["imdb_id"].notna(),
    ["title", "release_year", "imdb_id", "revenue", "budget"]
].copy()

numbers_parse = df.loc[
    df["title"].notna() & df["release_year"].notna(),
    ["title", "release_year", "imdb_id", "revenue", "budget", "revenue_adj_domestic"]
].copy()

imdb_parse.shape, mojo_parse.shape, numbers_parse.shape

((323, 8), (323, 5), (498, 6))

In [128]:
imdb_parse.shape, mojo_parse.shape, numbers_parse.shape

((323, 8), (323, 5), (498, 6))

### Датафреймы для парсинга

In [129]:
imdb_parse = df.loc[df["imdb_id"].notna(), ["title", "release_year", "imdb_id", "runtime", "vote_average_imdb", "vote_count_imdb", "budget", "genres"]].copy()

mojo_parse = df.loc[df["imdb_id"].notna(), ["title", "release_year", "imdb_id", "revenue", "budget"]].copy()

numbers_parse = df.loc[df["title"].notna() & df["release_year"].notna(), ["title", "release_year", "imdb_id", "revenue", "budget", "revenue_adj_domestic"]].copy()

### Вспомогательные функции

In [130]:
money_to_number = lambda x: float(re.sub("[^0-9]", "", x))

In [131]:
def text_norm(text):
    text = re.sub("\\s+", " ", str(text))
    return text

In [132]:
def imdb_duration_to_minutes(value):
    result = np.nan

    try:
        hours = re.findall("([0-9]+)H", value)
        minutes = re.findall("([0-9]+)M", value)

        result = 0

        if len(hours) > 0:
            result = result + float(hours[0]) * 60

        if len(minutes) > 0:
            result = result + float(minutes[0])

    except:
        result = np.nan

    return result

In [133]:
def imdb_duration_to_minutes(value):
    result = np.nan

    try:
        hours = re.findall("([0-9]+)H", value)
        minutes = re.findall("([0-9]+)M", value)

        result = 0

        if len(hours) > 0:
            result = result + float(hours[0]) * 60

        if len(minutes) > 0:
            result = result + float(minutes[0])

    except:
        result = np.nan

    return result

### IMDb: runtime, vote_average, vote_count, budget, genres

In [134]:
def parse_imdb(imdb_id):

    parsed_runtime = np.nan
    parsed_vote_average = np.nan
    parsed_vote_count = np.nan
    parsed_budget = np.nan
    parsed_genres = np.nan

    try:
        url = "https://www.imdb.com/title/" + imdb_id + "/"
        driver.get(url)
        time.sleep(2)
    except:
        return pd.Series([parsed_runtime, parsed_vote_average, parsed_vote_count, parsed_budget, parsed_genres])

    try:
        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        scripts = soup.find_all("script", {"type": "application/ld+json"})

        for script in scripts:
            try:
                data = json.loads(script.string)

                if type(data) == list:
                    data = data[0]

                if "duration" in data:
                    parsed_runtime = imdb_duration_to_minutes(data["duration"])

                if "aggregateRating" in data:
                    parsed_vote_average = float(data["aggregateRating"]["ratingValue"])
                    parsed_vote_count = float(data["aggregateRating"]["ratingCount"])

                if "genre" in data:
                    if type(data["genre"]) == list:
                        parsed_genres = ", ".join(data["genre"])

                    if type(data["genre"]) == str:
                        parsed_genres = data["genre"]

            except:
                pass

    except:
        parsed_runtime = np.nan
        parsed_vote_average = np.nan
        parsed_vote_count = np.nan
        parsed_genres = np.nan

    try:
        text = driver.find_element(By.TAG_NAME, "body").text
        text = text_norm(text)

        if pd.isna(parsed_runtime):
            runtime_result_1 = re.findall("Runtime ([0-9]+) hour[s]? ([0-9]+) minute[s]?", text, flags=re.I)
            runtime_result_2 = re.findall("Runtime ([0-9]+) minute[s]?", text, flags=re.I)

            if len(runtime_result_1) > 0:
                parsed_runtime = float(runtime_result_1[0][0]) * 60 + float(runtime_result_1[0][1])

            if len(runtime_result_2) > 0:
                parsed_runtime = float(runtime_result_2[0])

        if pd.isna(parsed_vote_average):
            rating_result = re.findall("IMDb RATING ([0-9.]+)\\s*/\\s*10 ([0-9.,KM]+)", text, flags=re.I)

            if len(rating_result) == 0:
                rating_result = re.findall("([0-9.]+)\\s*/\\s*10 ([0-9.,KM]+)", text, flags=re.I)

            if len(rating_result) > 0:
                parsed_vote_average = float(rating_result[0][0])
                parsed_vote_count = votes_to_number(rating_result[0][1])

        budget_result = re.findall("Budget\\s*\\$[0-9,]+", text, flags=re.I)

        if len(budget_result) > 0:
            parsed_budget = money_to_number(budget_result[0])

    except:
        parsed_budget = np.nan

    return pd.Series([parsed_runtime, parsed_vote_average, parsed_vote_count, parsed_budget, parsed_genres])

### Box Office Mojo: revenue + budget

In [135]:
def parse_mojo(imdb_id):
    url = "https://www.boxofficemojo.com/title/" + imdb_id + "/"
    driver.get(url)
    time.sleep(2)

    parsed_revenue_mojo = np.nan
    parsed_budget_mojo = np.nan

    try:
        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")

        block = soup.select_one(".mojo-performance-summary-table")

        if block is not None:
            block_text = block.get_text(" ", strip=True)
            result = re.findall("Worldwide\\s*(\\$[0-9,]+)", block_text, flags=re.I)

            if len(result) > 0:
                parsed_revenue_mojo = money_to_number(result[0])

        text = driver.find_element(By.TAG_NAME, "body").text
        text = text_norm(text)

        if pd.isna(parsed_revenue_mojo):
            result = re.findall("Worldwide\\s*\\$[0-9,]+", text, flags=re.I)

            if len(result) > 0:
                parsed_revenue_mojo = money_to_number(result[0])

        budget_result = re.findall("Budget\\s*\\$[0-9,]+", text, flags=re.I)

        if len(budget_result) > 0:
            parsed_budget_mojo = money_to_number(budget_result[0])

    except:
        parsed_revenue_mojo = np.nan
        parsed_budget_mojo = np.nan

    return pd.Series([parsed_revenue_mojo, parsed_budget_mojo])

### The Numbers: собираю URL

In [136]:
def make_numbers_title(title):
    title = str(title)

    title = title.replace(":", "")
    title = title.replace(",", "")
    title = title.replace(".", "")
    title = title.replace("'", "")
    title = title.replace("&", "and")
    title = title.replace("/", "-")
    title = title.replace("(", "")
    title = title.replace(")", "")

    words = title.split()

    if len(words) > 0:
        if words[0] == "The":
            words = words[1:] + [words[0]]

    title_url = "-".join(words)

    return title_url

In [137]:
def make_numbers_url_without_year(title):
    title_url = make_numbers_title(title)
    url = "https://www.the-numbers.com/movie/" + title_url + "#more"

    return url

In [138]:
def make_numbers_url_with_year(title, year):
    title_url = make_numbers_title(title)
    year = str(int(year))

    url = "https://www.the-numbers.com/movie/" + title_url + "-(" + year + ")#more"

    return url

In [139]:
def make_numbers_search_url(title, year):
    query = str(title) + " " + str(int(year))
    query = query.replace(" ", "+")

    url = "https://www.the-numbers.com/search?searchterm=" + query

    return url

### The Numbers: budget, revenue, Infl. Adj. Dom. BO

In [140]:
def parse_numbers_current_page():
    parsed_budget_numbers = np.nan
    parsed_revenue_numbers = np.nan
    parsed_revenue_adj_domestic = np.nan

    try:
        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")

        table = soup.select_one("#movie_finances")

        if table is not None:
            rows = table.find_all("tr")

            for row in rows:
                cells = row.find_all("td")
                cells_text = [cell.get_text(" ", strip=True) for cell in cells]

                if len(cells_text) > 1:
                    if "Worldwide Box Office" in cells_text[0]:
                        if "$" in cells_text[1]:
                            parsed_revenue_numbers = money_to_number(cells_text[1])

        rows = soup.find_all("tr")

        for row in rows:
            row_text = row.get_text(" ", strip=True)

            if "Production Budget" in row_text:
                result = re.findall("\\$[0-9,]+", row_text)

                if len(result) > 0:
                    parsed_budget_numbers = money_to_number(result[0])

            if "Infl. Adj. Dom. BO" in row_text:
                result = re.findall("\\$[0-9,]+", row_text)

                if len(result) > 0:
                    parsed_revenue_adj_domestic = money_to_number(result[0])

        text = soup.get_text(" ", strip=True)
        text = text_norm(text)

        if pd.isna(parsed_budget_numbers):
            result = re.findall("Production\\s+Budget:?\\s*(\\$[0-9,]+)", text, flags=re.I)

            if len(result) > 0:
                parsed_budget_numbers = money_to_number(result[0])

        if pd.isna(parsed_revenue_numbers):
            result = re.findall("Worldwide\\s+Box\\s+Office\\s*(\\$[0-9,]+)", text, flags=re.I)

            if len(result) > 0:
                parsed_revenue_numbers = money_to_number(result[0])

        if pd.isna(parsed_revenue_adj_domestic):
            result = re.findall("Infl\\.\\s*Adj\\.\\s*Dom\\.\\s*B\\.?O\\.?\\s*(\\$[0-9,]+)", text, flags=re.I)

            if len(result) > 0:
                parsed_revenue_adj_domestic = money_to_number(result[0])

    except:
        parsed_budget_numbers = np.nan
        parsed_revenue_numbers = np.nan
        parsed_revenue_adj_domestic = np.nan

    return pd.Series([parsed_budget_numbers, parsed_revenue_numbers, parsed_revenue_adj_domestic])

In [141]:
def parse_numbers_page(url):
    result = pd.Series([np.nan, np.nan, np.nan])

    try:
        driver.get(url)
        time.sleep(2)
        result = parse_numbers_current_page()

    except:
        try:
            driver.execute_script("window.stop();")
            time.sleep(1)
            result = parse_numbers_current_page()
        except:
            result = pd.Series([np.nan, np.nan, np.nan])

    return result

In [142]:
def parse_numbers_search(title, year):
    url = make_numbers_search_url(title, year)

    result = pd.Series([np.nan, np.nan, np.nan])

    try:
        driver.get(url)
        time.sleep(2)

        links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/movie/']")
        hrefs = []

        for link in links:
            href = link.get_attribute("href")
            link_text = link.text

            if href is not None:
                hrefs.append([href, link_text])

        title_first_word = str(title).split()[0].lower()

        for href, link_text in hrefs:
            if title_first_word in str(link_text).lower():
                result = parse_numbers_page(href)
                break

    except:
        result = pd.Series([np.nan, np.nan, np.nan])

    return result

In [143]:
def parse_numbers(title, year):
    url = make_numbers_url_without_year(title)
    result = parse_numbers_page(url)

    if result.isna().sum() == 3:
        url = make_numbers_url_with_year(title, year)
        result = parse_numbers_page(url)

    if result.isna().sum() == 3:
        result = parse_numbers_search(title, year)

    return result

### Запускаю Selenium

In [144]:
driver = webdriver.Chrome()
driver.set_page_load_timeout(40)

### Тест на 10 строках

In [24]:
imdb_test = imdb_parse.head(10).copy()
mojo_test = mojo_parse.head(10).copy()
numbers_test = numbers_parse.head(10).copy()

### IMDb тест

In [25]:
tqdm.pandas(desc="Тест: IMDb")

imdb_test[["parsed_runtime_imdb", "parsed_vote_average_imdb", "parsed_vote_count_imdb", "parsed_budget_imdb", "parsed_genres_imdb"]] = imdb_test["imdb_id"].progress_apply(parse_imdb)

imdb_test[["title", "imdb_id", "runtime", "vote_average_imdb", "vote_count_imdb", "budget", "genres", "parsed_runtime_imdb", "parsed_vote_average_imdb", "parsed_vote_count_imdb", "parsed_budget_imdb", "parsed_genres_imdb"]]

Тест: IMDb:   0%|          | 0/10 [00:00<?, ?it/s]

,title,imdb_id,runtime,vote_average_imdb,vote_count_imdb,budget,genres,parsed_runtime_imdb,parsed_vote_average_imdb,parsed_vote_count_imdb,parsed_budget_imdb,parsed_genres_imdb
3,The Hunter,tt1703148,102.0,6.7,42712.0,NaN,"Drama, Thriller, Adventure",102.0,6.7,42744.0,NaN,"Adventure, Drama, Thriller"
4,The Do-Over,tt4769836,108.0,5.7,57047.0,NaN,"Action, Adventure, Comedy",108.0,5.7,57166.0,NaN,"Action, Adventure, Comedy"
5,Hyena Road,tt4034452,120.0,6.5,10004.0,NaN,"War, Drama, Thriller, Action",120.0,6.5,10027.0,NaN,"Drama, War"
6,City of God: 10 Years Later,tt4103686,69.0,6.4,1581.0,NaN,Documentary,70.0,6.4,1581.0,130000.0,Documentary
7,Sand Castle,tt2582576,113.0,6.3,30813.0,NaN,"War, Action, Drama",113.0,6.3,30877.0,NaN,"Action, Drama, War"
8,Sandy Wexler,tt5893332,130.0,5.3,20531.0,NaN,Comedy,130.0,5.3,20568.0,24300000.0,"Comedy, Drama, Music"
9,Back and Forth,tt5046504,100.0,5.7,559.0,NaN,"Romance, Comedy",100.0,5.7,559.0,NaN,"Comedy, Drama, Romance"
10,Happily Married,tt4006906,91.0,5.2,306.0,NaN,"Romance, Comedy",91.0,5.2,307.0,NaN,"Comedy, Romance"
12,Stranger Things,tt4574334,0.0,8.6,1695741.0,NaN,"Science Fiction, Horror, Mystery, Drama",NaN,8.6,1700749.0,NaN,"Drama, Fantasy, Horror"
19,The Discovery,tt5155780,102.0,6.2,40485.0,NaN,"Drama, Romance, Science Fiction, Thriller, Mys...",102.0,6.2,40562.0,NaN,"Drama, Romance, Sci-Fi"


### BOX mojo тест

In [26]:
tqdm.pandas(desc="Тест: Box Office Mojo")

mojo_test[["parsed_revenue_mojo", "parsed_budget_mojo"]] = mojo_test["imdb_id"].progress_apply(parse_mojo)

mojo_test[["title", "imdb_id", "revenue", "budget", "parsed_revenue_mojo", "parsed_budget_mojo"]]

Тест: Box Office Mojo:   0%|          | 0/10 [00:00<?, ?it/s]

,title,imdb_id,revenue,budget,parsed_revenue_mojo,parsed_budget_mojo
3,The Hunter,tt1703148,176669.0,NaN,1680778.0,NaN
4,The Do-Over,tt4769836,NaN,NaN,NaN,NaN
5,Hyena Road,tt4034452,87768.0,NaN,87768.0,NaN
6,City of God: 10 Years Later,tt4103686,NaN,NaN,NaN,NaN
7,Sand Castle,tt2582576,NaN,NaN,NaN,NaN
8,Sandy Wexler,tt5893332,NaN,NaN,NaN,NaN
9,Back and Forth,tt5046504,NaN,NaN,485619.0,NaN
10,Happily Married,tt4006906,NaN,NaN,NaN,NaN
12,Stranger Things,tt4574334,NaN,NaN,NaN,NaN
19,The Discovery,tt5155780,NaN,NaN,NaN,NaN


### The Numbers test

In [28]:
tqdm.pandas(desc="Тест: The Numbers")

numbers_test[["parsed_budget_numbers", "parsed_revenue_numbers", "parsed_revenue_adj_domestic"]] = numbers_test.progress_apply(lambda x: parse_numbers(x["title"], x["release_year"]), axis=1)

numbers_test[["title", "release_year", "revenue", "budget", "revenue_adj_domestic", "parsed_budget_numbers", "parsed_revenue_numbers", "parsed_revenue_adj_domestic"]]

Тест: The Numbers:   0%|          | 0/10 [00:00<?, ?it/s]

,title,release_year,revenue,budget,revenue_adj_domestic,parsed_budget_numbers,parsed_revenue_numbers,parsed_revenue_adj_domestic
0,White Chicks,2004,113086475.0,37000000.0,NaN,20000000.0,111448997.0,133175844.0
1,Lucky Number Slevin,2006,56308881.0,27000000.0,NaN,27000000.0,55495466.0,41075687.0
2,Death Note,2006,29667169.0,20000000.0,NaN,NaN,NaN,NaN
3,The Hunter,2011,176669.0,NaN,NaN,NaN,NaN,69995023.0
4,The Do-Over,2016,NaN,NaN,NaN,22000000.0,88180367.0,29080367.0
5,Hyena Road,2015,87768.0,NaN,NaN,NaN,NaN,1973.0
6,City of God: 10 Years Later,2013,NaN,NaN,NaN,NaN,NaN,NaN
7,Sand Castle,2017,NaN,NaN,NaN,NaN,NaN,NaN
8,Sandy Wexler,2017,NaN,NaN,NaN,NaN,NaN,NaN
9,Back and Forth,2016,NaN,NaN,NaN,NaN,NaN,NaN


### Полный парсинг IMDb

In [118]:
tqdm.pandas(desc="Парсим IMDb")

imdb_parse[["parsed_runtime_imdb", "parsed_vote_average_imdb", "parsed_vote_count_imdb", "parsed_budget_imdb", "parsed_genres_imdb"]] = imdb_parse["imdb_id"].progress_apply(parse_imdb)

imdb_parse.to_csv("parsed_imdb.csv", index=True)

Парсим IMDb:   0%|          | 0/323 [00:00<?, ?it/s]

### Полный парсинг budget с Box Office Mojo

In [119]:
tqdm.pandas(desc="Парсим Box Office Mojo")

mojo_parse[["parsed_revenue_mojo", "parsed_budget_mojo"]] = mojo_parse["imdb_id"].progress_apply(parse_mojo)

mojo_parse.to_csv("parsed_mojo.csv", index=True)

Парсим Box Office Mojo:   0%|          | 0/323 [00:00<?, ?it/s]

### Полный парсинг The Numbers

In [120]:
numbers_parse["parsed_budget_numbers"] = np.nan
numbers_parse["parsed_revenue_numbers"] = np.nan
numbers_parse["parsed_revenue_adj_domestic"] = np.nan

In [145]:
step = 50

for start in range(0, numbers_parse.shape[0], step):
    finish = start + step

    numbers_part = numbers_parse.iloc[start:finish].copy()

    tqdm.pandas(desc="Парсим The Numbers " + str(start) + "-" + str(finish))

    numbers_parse.loc[
        numbers_part.index,
        ["parsed_budget_numbers", "parsed_revenue_numbers", "parsed_revenue_adj_domestic"]
    ] = numbers_part.progress_apply(
        lambda x: parse_numbers(x["title"], x["release_year"]),
        axis=1
    )

    numbers_parse.to_csv("parsed_numbers.csv", index=True)

Парсим The Numbers 0-50:   0%|          | 0/50 [00:00<?, ?it/s]

Парсим The Numbers 50-100:   0%|          | 0/50 [00:00<?, ?it/s]

Парсим The Numbers 100-150:   0%|          | 0/50 [00:00<?, ?it/s]

Парсим The Numbers 150-200:   0%|          | 0/50 [00:00<?, ?it/s]

Парсим The Numbers 200-250:   0%|          | 0/50 [00:00<?, ?it/s]

Парсим The Numbers 250-300:   0%|          | 0/50 [00:00<?, ?it/s]

Парсим The Numbers 300-350:   0%|          | 0/50 [00:00<?, ?it/s]

Парсим The Numbers 350-400:   0%|          | 0/50 [00:00<?, ?it/s]

Парсим The Numbers 400-450:   0%|          | 0/50 [00:00<?, ?it/s]

Парсим The Numbers 450-500:   0%|          | 0/48 [00:00<?, ?it/s]

### Закрываю браузер

In [ ]:
driver.quit()